## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

## Extracting TerraClimate Data Using API Calls

The API-based method allows us to efficiently access **TerraClimate** data for specific regions and time periods through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

Through the API, we can extract climate variables such as **Potential Evapotranspiration (PET)**, which represents the atmospheric demand for water. This variable provides important insights into surface moisture balance and helps improve the accuracy of water quality modeling.

This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.

### Loading and Mapping TerraClimate Data

This section demonstrates how **TerraClimate climate variables**, such as **Potential Evapotranspiration (PET)**, are loaded and mapped to sampling locations.

- The **load_terraclimate_dataset** function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.
- The **filterg** function filters the dataset for the desired time range (2011–2015) and the spatial extent corresponding to the study region. The resulting data is converted into a pandas DataFrame with standardized column names.
- The **assign_nearest_climate** function maps each sampling location to its **nearest TerraClimate grid point** using a KD-tree and assigns the climate variable values corresponding to the closest timestamp.

This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.

In [2]:

def load_terraclimate_dataset(varsagg):
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    ds = ds[varsagg]

    ds = ds.sel(
        time=slice("2010-08-30", "2016-01-02"),
        lat=slice(-21.72, -35.18),
        lon=slice(14.97, 32.79)
    )

   
    return ds

In [4]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final

In [ ]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

### Extracting features for the training dataset

In [5]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
#display(Water_Quality_df.head(5))

Validation_df=pd.read_csv('submission_template.csv')
#display(Validation_df.head(5))

TerClim_df = Water_Quality_df[
    ['Latitude','Longitude','Sample Date']
].copy()

TerCVal_df = Validation_df[
    ['Latitude','Longitude','Sample Date']
].copy()

In [ ]:
varsagg = ['pet', 'ppt', 'q', 'soil', 'tmax', 'tmin', 'aet']

ds_pre = load_terraclimate_dataset(varsagg)

In [ ]:
ds_pre = ds_pre.chunk({'time': 12, 'lat': 1024, 'lon': 1024})

ds_pre.to_zarr("terraclimate_subset.zarr", mode="w")

ds = xr.open_zarr("terraclimate_subset.zarr")

In [ ]:
#ds.head()

In [ ]:

#varsagg = ['pet', 'ppt', 'q', 'soil', 'tmax', 'tmin', 'aet']
for var in varsagg:
    tc_param = filterg(ds, var)
    TrainAux = assign_nearest_climate(Water_Quality_df, tc_param, var)
    ValAux = assign_nearest_climate(Validation_df, tc_param, var)
    TerClim_df[var] = TrainAux[var].values
    TerCVal_df[var] = ValAux[var].values
    print(f"{var} complete for Val")

In [9]:
#Create the lags and rolling windows, input needs to have the correct fields! 
def extract_tc_lags(
    ds,
    wq_df,
    vars_tc,
    lags=(0, 1, 2)
):

    tc_df = ds.to_dataframe().reset_index()

    tc_df['year'] = tc_df['time'].dt.year
    tc_df['month'] = tc_df['time'].dt.month

    wq_df['Date'] = pd.to_datetime(wq_df['Sample Date'], dayfirst=True,
    errors='raise')
    wq_df['year'] = wq_df['Date'].dt.year
    wq_df['month'] = wq_df['Date'].dt.month
    
    from scipy.spatial import cKDTree

    # Nearest grid (solo una vez)
    grid = tc_df[['lat', 'lon']].drop_duplicates().reset_index(drop=True)

    tree = cKDTree(np.radians(grid[['lat', 'lon']].values))
    _, idx = tree.query(np.radians(wq_df[['Latitude', 'Longitude']].values), k=1)

    nearest = grid.iloc[idx].reset_index(drop=True)

    features = []

    for i, row in wq_df.iterrows():
        lat = nearest.loc[i, 'lat']
        lon = nearest.loc[i, 'lon']
        y = row['year']
        m = row['month']

        row_feat = {}

        for lag in lags:
            y_l, m_l = shift_month(y, m, lag)

            subset = tc_df[
                (tc_df['lat'] == lat) &
                (tc_df['lon'] == lon) &
                (tc_df['year'] == y_l) &
                (tc_df['month'] == m_l)
            ]

            for var in vars_tc:
                col = var if lag == 0 else f"{var}_lag{lag}"
                row_feat[col] = subset[var].values[0] if not subset.empty else np.nan

        features.append(row_feat)

    return pd.DataFrame(features)


def extract_tc_rollsum(ds, wq_df, vars_tc, rolls=(3,6,12)):

    tc_df = ds.to_dataframe().reset_index()

    # Crear year, month y clave ym UNA VEZ
    tc_df['year'] = tc_df['time'].dt.year
    tc_df['month'] = tc_df['time'].dt.month
    tc_df['ym'] = tc_df['year']*100 + tc_df['month']

    wq_df = wq_df.copy()
    wq_df['Date'] = pd.to_datetime(wq_df['Sample Date'], dayfirst=True, errors='raise')
    wq_df['year'] = wq_df['Date'].dt.year
    wq_df['month'] = wq_df['Date'].dt.month
    wq_df['ym'] = wq_df['year']*100 + wq_df['month']

    from scipy.spatial import cKDTree

    # Nearest grid
    grid = tc_df[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(np.radians(grid[['lat', 'lon']].values))
    _, idx = tree.query(np.radians(wq_df[['Latitude', 'Longitude']].values), k=1)
    nearest = grid.iloc[idx].reset_index(drop=True)

    grouped = tc_df.groupby(['lat','lon'])

    features = []

    for i in range(len(wq_df)):

        lat = nearest.loc[i, 'lat']
        lon = nearest.loc[i, 'lon']
        y = wq_df.loc[i, 'year']
        m = wq_df.loc[i, 'month']

        row_feat = {}

        # Obtener subtabla solo para ese punto
        try:
            subset_loc = grouped.get_group((lat,lon))
        except KeyError:
            features.append({f"{v}_roll{r}_sum": np.nan for v in vars_tc for r in rolls})
            continue

        for roll in rolls:

            months_to_sum = [
                shift_month(y, m, k)[0]*100 + shift_month(y, m, k)[1]
                for k in range(roll)
            ]

            subset = subset_loc[subset_loc['ym'].isin(months_to_sum)]

            for var in vars_tc:
                row_feat[f"{var}_roll{roll}_sum"] = (
                    subset[var].sum() if not subset.empty else np.nan
                )

        features.append(row_feat)

    return pd.DataFrame(features)


def extract_tc_rollmean(ds, wq_df, vars_tc, rolls=(3,6,12)):

    tc_df = ds.to_dataframe().reset_index()

    # Crear year, month y clave ym UNA VEZ
    tc_df['year'] = tc_df['time'].dt.year
    tc_df['month'] = tc_df['time'].dt.month
    tc_df['ym'] = tc_df['year']*100 + tc_df['month']

    wq_df = wq_df.copy()
    wq_df['Date'] = pd.to_datetime(wq_df['Sample Date'], dayfirst=True, errors='raise')
    wq_df['year'] = wq_df['Date'].dt.year
    wq_df['month'] = wq_df['Date'].dt.month
    wq_df['ym'] = wq_df['year']*100 + wq_df['month']

    from scipy.spatial import cKDTree

    # Nearest grid
    grid = tc_df[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(np.radians(grid[['lat', 'lon']].values))
    _, idx = tree.query(np.radians(wq_df[['Latitude', 'Longitude']].values), k=1)
    nearest = grid.iloc[idx].reset_index(drop=True)

    grouped = tc_df.groupby(['lat','lon'])

    features = []

    for i in range(len(wq_df)):

        lat = nearest.loc[i, 'lat']
        lon = nearest.loc[i, 'lon']
        y = wq_df.loc[i, 'year']
        m = wq_df.loc[i, 'month']

        row_feat = {}

        # Obtener subtabla solo para ese punto
        try:
            subset_loc = grouped.get_group((lat,lon))
        except KeyError:
            features.append({f"{v}_roll{r}_mean": np.nan for v in vars_tc for r in rolls})
            continue

        for roll in rolls:

            months_to_sum = [
                shift_month(y, m, k)[0]*100 + shift_month(y, m, k)[1]
                for k in range(roll)
            ]

            subset = subset_loc[subset_loc['ym'].isin(months_to_sum)]

            for var in vars_tc:
                row_feat[f"{var}_roll{roll}_mean"] = (
                    subset[var].mean() if not subset.empty else np.nan
                )

        features.append(row_feat)

    return pd.DataFrame(features)



def shift_month(year, month, lag):
    month -= lag
    while month <= 0:
        month += 12
        year -= 1
    return year, month


In [ ]:
varlag = ['pet', 'ppt','q', 'tmax', 'tmin', 'aet']

tc_lags = extract_tc_lags(
    ds,TerClim_df,
    varlag,
    lags=(1, 2, 3)
)
print('lags de TerClim_df ready!')

val_lags = extract_tc_lags(
    ds,TerCVal_df,
    varlag,
    lags=(1, 2, 3)
)
print('lags de TerCVal_df ready!')

In [ ]:
varsum = ['ppt','q']
rolls=(3, 6, 12)

tc_sumroll = extract_tc_rollsum(
    ds,TerClim_df,
    varsum,
    rolls
)

print('rollsum de TerClim_df ready!')

val_sumroll = extract_tc_rollsum(
    ds,TerCVal_df,
    varsum,
    rolls
)

print('rollsum de TerCVal_df ready!')

varmean = ['tmin','tmax', 'soil']

tc_meanroll = extract_tc_rollmean(
    ds,TerClim_df,
    varmean,
    rolls
)


print('rollmean de TerClim_df ready!')

val_meanroll = extract_tc_rollmean(
    ds,TerCVal_df,
    varmean,
    rolls
)

print('rollmean de TerCVal_df ready!')

In [ ]:
final_tc_df = pd.concat(
    [TerClim_df.reset_index(drop=True), tc_lags, tc_sumroll, tc_meanroll],
    axis=1
)
final_val_df = pd.concat(
    [TerCVal_df.reset_index(drop=True), val_lags, val_sumroll, val_meanroll],
    axis=1
)

In [ ]:
display(final_tc_df.info())
display(final_val_df.info())

final_tc_df = final_tc_df.drop(columns=['Date','year'] )
final_val_df = final_val_df.drop(columns=['Date','year'] )

In [ ]:

row_id = 1234
row = final_tc_df.loc[row_id]
display(row)

subset = ds.sel(
    lat=slice(-27.30, -27.32),
    lon=slice(28.6, 29.61)).sel( 
    time='09-05-2011', method="nearest"
)
subset_df = subset.to_dataframe().reset_index()
subset_df


In [ ]:
#final_val_df.info()

In [ ]:
final_tc_df.to_csv("/tmp/terraclimate_features_training_v3.csv",index = False)

session.sql(f"""
    PUT file:///tmp/terraclimate_features_training_v3.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

### Extracting features for the validation dataset

In [ ]:
final_val_df.to_csv("/tmp/terraclimate_features_validation_v3.csv",index = False)

session.sql(f"""
    PUT file:///tmp/terraclimate_features_validation_v3.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()


print("File saved! Refresh the browser to see the files in the sidebar")